# Figshare UniversalXAS experiment grid and element-wise fine-tuning

This notebook runs a controlled model-selection study on the unscaled Figshare splits and fixed 64D features. It tests **81 UniversalXAS configurations**: three loss-balance strengths, three dropout rates, three learning rates, and three random seeds. For each element, it validation-ranks all candidates and fine-tunes the top three using four adaptation choices (full-head vs output-layer-only, each at two learning rates): up to **96 fine-tuned runs**.

The loss-balance sweep compares raw MSE, partial baseline-relative weighting, and equal-weight baseline-relative MSE. Targets themselves are never rescaled. The final validation and test eta tables and charts compare the existing raw-MSE controls with the validation-selected UniversalXAS and Tuned-UniversalXAS models. Test metrics are calculated only after every selection is complete.

## Remote notebook setup

Run this notebook in the OmniXAS Python environment from the repository or `tutorial_omnixas` directory. If the notebook server stores the repository or data elsewhere, set `OMNIXAS_REPO_ROOT`, `OMNIXAS_DATA_ROOT`, `OMNIXAS_SPLIT_ROOT`, and/or `OMNIXAS_FEATURE_RUN_DIR` before starting Jupyter. The search grid and fine-tuning options are defined near the top of the setup cell and can be adjusted there. Set `OMNIXAS_HEAD_SWEEP_DIR` to a fresh folder to preserve or compare sweep runs.


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
repo_env = os.environ.get('OMNIXAS_REPO_ROOT')
if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    candidates = [cwd, *cwd.parents]
    REPO_ROOT = next((p for p in candidates if (p / 'omnixas').is_dir() and (p / 'tutorial_omnixas').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Cannot find the OmniXAS repository. Set OMNIXAS_REPO_ROOT to its path.')
sys.path.insert(0, str(REPO_ROOT))
DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser().resolve()
SPLIT_ROOT = Path(os.environ.get('OMNIXAS_SPLIT_ROOT', DATA_ROOT / 'figshare_141_splits_unscaled')).expanduser().resolve()
FEATURE_RUN_DIR = Path(os.environ.get(
    'OMNIXAS_FEATURE_RUN_DIR',
    REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_pipeline' / 'figshare_141_unscaled_seed42',
)).expanduser().resolve()
FEATURE_ROOT = FEATURE_RUN_DIR / 'features'
OUTPUT_DIR = Path(os.environ.get(
    'OMNIXAS_HEAD_SWEEP_DIR', FEATURE_RUN_DIR / 'figshare_universalxas_experiment_grid',
)).expanduser().resolve()
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Controlled UniversalXAS grid: 3 loss weights x 3 dropout rates x 3 learning rates x 3 seeds = 81 runs.
UNIVERSAL_SEEDS = [42, 43, 44]
LOSS_BALANCE_POWERS = [0.0, 0.5, 1.0]  # 0=raw MSE, 0.5=partial weighting, 1=full baseline-relative weighting
UNIVERSAL_DROPOUTS = [0.0, 0.10, 0.25]
UNIVERSAL_LEARNING_RATES = [3e-4, 5e-4, 1e-3]
TOP_UNIVERSAL_CANDIDATES_PER_ELEMENT = 3
FINE_TUNE_CONFIGS = [
    {'strategy': 'full_head', 'learning_rate': 1e-4},
    {'strategy': 'full_head', 'learning_rate': 3e-4},
    {'strategy': 'output_layer_only', 'learning_rate': 3e-4},
    {'strategy': 'output_layer_only', 'learning_rate': 1e-3},
]
N_UNIVERSAL_CANDIDATES = (len(UNIVERSAL_SEEDS) * len(LOSS_BALANCE_POWERS)
                          * len(UNIVERSAL_DROPOUTS) * len(UNIVERSAL_LEARNING_RATES))
N_FINE_TUNE_CANDIDATES = (len(ELEMENTS) * TOP_UNIVERSAL_CANDIDATES_PER_ELEMENT
                          * len(FINE_TUNE_CONFIGS))

for required in [
    SPLIT_ROOT / 'report.json', SPLIT_ROOT / 'manifest.csv',
    FEATURE_RUN_DIR / 'RUN_COMPLETE.json', FEATURE_RUN_DIR / 'best_encoder.ckpt',
    FEATURE_ROOT,
]:
    if not required.exists():
        raise FileNotFoundError(f'Required remote notebook input is missing: {required}')
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Output already exists: {OUTPUT_DIR}. Set OMNIXAS_HEAD_SWEEP_DIR to a fresh directory to preserve prior results.')

report = json.loads((SPLIT_ROOT / 'report.json').read_text(encoding='utf-8'))
if report.get('intensity_normalization') != 'none (factor 1.0; no xsedge-derived scale)':
    raise ValueError(f'Expected unscaled Figshare targets; found {report.get("intensity_normalization")!r}')
manifest = pd.read_csv(SPLIT_ROOT / 'manifest.csv')
if not np.allclose(pd.to_numeric(manifest['target_scale_factor']), 1.0, rtol=0, atol=0):
    raise ValueError('Split manifest contains a target scale other than 1.0')

# Confirm that exported features, source targets, IDs, and manifest rows agree.
data = {}
for element in ELEMENTS:
    for split in SPLITS:
        task = f'{element}_FEFF'
        ids_path = SPLIT_ROOT / 'material_id_and_site' / f'{task}_{split}.txt'
        y_path = SPLIT_ROOT / 'ml_data' / f'{task}_{split}_y.txt'
        x_feature_path = FEATURE_ROOT / f'{task}_{split}_X.txt'
        y_feature_path = FEATURE_ROOT / f'{task}_{split}_y.txt'
        ids = [line.strip() for line in ids_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
        x = np.atleast_2d(np.loadtxt(x_feature_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        y_feature = np.atleast_2d(np.loadtxt(y_feature_path, dtype=np.float32))
        rows = manifest[(manifest.element == element) & (manifest.split == split)].sort_values('row_index')
        if rows.material_id_and_site.tolist() != ids:
            raise ValueError(f'Manifest/ID order mismatch for {task} {split}')
        if x.shape != (len(ids), 64) or y.shape != (len(ids), 141) or y_feature.shape != y.shape:
            raise ValueError(f'Unexpected feature/target dimensions for {task} {split}: X={x.shape}, y={y.shape}, feature-y={y_feature.shape}')
        if not np.isfinite(x).all() or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError(f'Non-finite or negative values in {task} {split}')
        if not np.allclose(y, y_feature, rtol=1e-6, atol=1e-7):
            raise ValueError(f'Cached features were generated from different targets for {task} {split}')
        data[(element, split)] = {'ids': ids, 'X': x, 'y': y}

# Fixed train-only reference values for the loss; no validation/test values enter training.
TRAIN_MEANS = {}
LOSS_SCALES = {}
for element in ELEMENTS:
    y_train = data[(element, 'train')]['y']
    train_mean = y_train.mean(axis=0, keepdims=True)
    per_spectrum_baseline = np.mean((y_train - train_mean) ** 2, axis=1)
    TRAIN_MEANS[element] = train_mean.astype(np.float32)
    LOSS_SCALES[element] = float(np.median(per_spectrum_baseline))
    if not np.isfinite(LOSS_SCALES[element]) or LOSS_SCALES[element] <= 0:
        raise ValueError(f'Invalid train-only loss scale for {element}: {LOSS_SCALES[element]}')

# Existing checkpoints are the raw-MSE controls from the completed Figshare run.
OLD_UNIVERSAL = FEATURE_RUN_DIR / 'heads' / 'universalXAS' / 'best.pt'
OLD_TUNED = {e: FEATURE_RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{e}_FEFF' / 'best.pt' for e in ELEMENTS}
missing_heads = [p for p in [OLD_UNIVERSAL, *OLD_TUNED.values()] if not p.is_file()]
if missing_heads:
    raise FileNotFoundError('Completed raw-MSE comparison heads are missing:\n' + '\n'.join(map(str, missing_heads)))

OUTPUT_DIR.mkdir(parents=True)
config = {
    'dataset': str(SPLIT_ROOT),
    'feature_run': str(FEATURE_RUN_DIR),
    'source_encoder_checkpoint': str(FEATURE_RUN_DIR / 'best_encoder.ckpt'),
    'target_policy': report.get('target_policy'),
    'intensity_normalization': report.get('intensity_normalization'),
    'objective': 'equal mean across elements of per-element batch MSE divided by train-only baseline MSE raised to loss_balance_power',
    'universal_seeds': UNIVERSAL_SEEDS,
    'loss_balance_powers': LOSS_BALANCE_POWERS,
    'universal_dropouts': UNIVERSAL_DROPOUTS,
    'universal_learning_rates': UNIVERSAL_LEARNING_RATES,
    'universal_candidate_count': N_UNIVERSAL_CANDIDATES,
    'top_universal_candidates_per_element_to_fine_tune': TOP_UNIVERSAL_CANDIDATES_PER_ELEMENT,
    'fine_tune_configs': FINE_TUNE_CONFIGS,
    'fine_tune_candidate_count': N_FINE_TUNE_CANDIDATES,
    'universal_lr': 5e-4,
    'fine_tune_lr': 3e-4,
    'checkpoint_selection': 'per-element validation eta; test evaluated after selection only',
    'device': str(DEVICE),
}
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
print(f'Repository: {REPO_ROOT}')
print(f'Dataset: {SPLIT_ROOT}')
print(f'Feature run: {FEATURE_RUN_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')
print('Targets are verified unscaled; feature/target/ID rows match for all 24 element-splits.')
print(f'Planned search: {N_UNIVERSAL_CANDIDATES} UniversalXAS candidates and up to {N_FINE_TUNE_CANDIDATES} fine-tunes.')

## Search design and selection rules

All UniversalXAS candidates use the same balanced-with-replacement training rows per element and the same train/validation split. The 81-run grid changes one of three interpretable choices in a full factorial design:

- **Loss balance power** `alpha`: the per-element loss is `MSE / (train_baseline ** alpha)`. `alpha=0` is raw MSE, `0.5` is partial normalization, and `1` is the equal-weight relative-MSE objective. The spectra remain unchanged.
- **Dropout:** `0.0`, `0.10`, or `0.25`.
- **Universal learning rate:** `3e-4`, `5e-4`, or `1e-3`.

Each combination is repeated with three seeds. Candidate ranking is per element by validation eta. The top three UniversalXAS sources per element are then adapted with four fine-tuning choices: full-head or output-layer-only training, each at two learning rates. The best tuned run per element is selected by validation eta. This is an exploratory hyperparameter search: report the selected test eta once, and do not use it to revise the choices.


In [ ]:
from itertools import product
from omnixas.model.m3gnet_xas import XASSpectralHead

UNIVERSAL_MAX_EPOCHS = 800
UNIVERSAL_PATIENCE = 60
FINE_TUNE_MAX_EPOCHS = 1000
FINE_TUNE_PATIENCE = 60
BATCH_SIZE = 4096


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cpu_state_dict(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}


def evaluate_element(head, element, split):
    item = data[(element, split)]
    x = torch.as_tensor(item['X'], dtype=torch.float32, device=DEVICE)
    y = torch.as_tensor(item['y'], dtype=torch.float32, device=DEVICE)
    head.eval()
    with torch.inference_mode():
        prediction = head(x)
        errors = (prediction - y).square().mean(dim=1).cpu().numpy()
    return errors


def validation_summary(head):
    scores = {}
    for element in ELEMENTS:
        errors = evaluate_element(head, element, 'val')
        median_mse = float(np.median(errors))
        baseline_errors = np.mean((data[(element, 'val')]['y'] - TRAIN_MEANS[element]) ** 2, axis=1)
        baseline_median = float(np.median(baseline_errors))
        scores[element] = {
            'median_mse': median_mse,
            'baseline_median_mse': baseline_median,
            'eta': baseline_median / max(median_mse, 1e-12),
            'relative_median_mse': median_mse / LOSS_SCALES[element],
        }
    return scores


def balanced_training_arrays(seed):
    rng = np.random.default_rng(seed)
    rows_per_element = max(len(data[(element, 'train')]['y']) for element in ELEMENTS)
    xs, ys, task_ids = [], [], []
    for task_index, element in enumerate(ELEMENTS):
        item = data[(element, 'train')]
        selected = rng.integers(0, len(item['y']), size=rows_per_element)
        xs.append(item['X'][selected])
        ys.append(item['y'][selected])
        task_ids.append(np.full(rows_per_element, task_index, dtype=np.int64))
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(task_ids)


def short_float(value):
    return f'{value:g}'.replace('.', 'p').replace('-', 'm')


def train_universal_candidate(loss_power, dropout, learning_rate, seed):
    set_seed(seed)  # matched seed gives comparable initial weights across grid settings
    experiment_id = f'alpha{short_float(loss_power)}_drop{short_float(dropout)}_lr{short_float(learning_rate)}_seed{seed}'
    seed_dir = OUTPUT_DIR / 'universal_candidates' / experiment_id
    task_ckpt_dir = seed_dir / 'best_by_element'
    task_ckpt_dir.mkdir(parents=True, exist_ok=False)
    train_x, train_y, train_task = balanced_training_arrays(seed)
    train_x = torch.as_tensor(train_x, dtype=torch.float32, device=DEVICE)
    train_y = torch.as_tensor(train_y, dtype=torch.float32, device=DEVICE)
    train_task = torch.as_tensor(train_task, dtype=torch.long, device=DEVICE)
    loss_scale = torch.as_tensor([LOSS_SCALES[e] ** loss_power for e in ELEMENTS], dtype=torch.float32, device=DEVICE)
    head = XASSpectralHead(dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    best_task_relative = {e: float('inf') for e in ELEMENTS}
    best_macro_relative = float('inf')
    stale = 0
    history = []
    progress = tqdm(range(UNIVERSAL_MAX_EPOCHS), desc=f'UniversalXAS {experiment_id}', unit='epoch')
    for epoch in progress:
        head.train()
        order = torch.randperm(len(train_x), device=DEVICE)
        for batch_indices in order.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            prediction = head(train_x[batch_indices])
            per_row_mse = (prediction - train_y[batch_indices]).square().mean(dim=1)
            batch_tasks = train_task[batch_indices]
            by_element = []
            for task_index in range(len(ELEMENTS)):
                mask = batch_tasks == task_index
                if not mask.any():
                    raise RuntimeError(f'Universal batch lacks {ELEMENTS[task_index]} examples')
                by_element.append(per_row_mse[mask].mean() / loss_scale[task_index])
            loss = torch.stack(by_element).mean()
            loss.backward()
            optimizer.step()

        scores = validation_summary(head)
        macro_relative = float(np.mean([scores[e]['relative_median_mse'] for e in ELEMENTS]))
        history.append({'epoch': epoch + 1, 'macro_relative_median_mse': macro_relative,
                        **{f'{e}_val_eta': scores[e]['eta'] for e in ELEMENTS}})
        for element in ELEMENTS:
            relative = scores[element]['relative_median_mse']
            if relative < best_task_relative[element]:
                best_task_relative[element] = relative
                torch.save({
                    'state_dict': cpu_state_dict(head), 'seed': seed, 'epoch': epoch + 1,
                    'element': element, 'val_eta': scores[element]['eta'],
                    'val_median_mse': scores[element]['median_mse'],
                    'val_relative_median_mse': relative,
                    'experiment_id': experiment_id, 'loss_balance_power': loss_power,
                    'dropout': dropout, 'learning_rate': learning_rate,
                }, task_ckpt_dir / f'{element}.pt')
        if macro_relative < best_macro_relative:
            best_macro_relative, stale = macro_relative, 0
            torch.save({
                'state_dict': cpu_state_dict(head), 'seed': seed, 'epoch': epoch + 1,
                'macro_relative_median_mse': macro_relative, 'experiment_id': experiment_id,
                'loss_balance_power': loss_power, 'dropout': dropout, 'learning_rate': learning_rate,
            }, seed_dir / 'best_macro.pt')
        else:
            stale += 1
        scheduler.step(macro_relative)
        progress.set_postfix(val_rel=f'{macro_relative:.4f}', best=f'{best_macro_relative:.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.1e}')
        if stale >= UNIVERSAL_PATIENCE:
            break
    progress.close()
    pd.DataFrame(history).to_csv(seed_dir / 'validation_history.csv', index=False)
    records = []
    for element in ELEMENTS:
        checkpoint = task_ckpt_dir / f'{element}.pt'
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        records.append({
            'element': element, 'seed': seed, 'epoch': saved['epoch'],
            'experiment_id': experiment_id, 'loss_balance_power': loss_power,
            'dropout': dropout, 'learning_rate': learning_rate,
            'val_median_mse': saved['val_median_mse'],
            'val_relative_median_mse': saved['val_relative_median_mse'],
            'val_eta': saved['val_eta'], 'checkpoint': str(checkpoint),
        })
    return records


universal_val_records = []
for loss_power, dropout, learning_rate, seed in product(
    LOSS_BALANCE_POWERS, UNIVERSAL_DROPOUTS, UNIVERSAL_LEARNING_RATES, UNIVERSAL_SEEDS
):
    universal_val_records.extend(train_universal_candidate(loss_power, dropout, learning_rate, seed))
universal_val = pd.DataFrame(universal_val_records)
if len(universal_val) != N_UNIVERSAL_CANDIDATES * len(ELEMENTS):
    raise RuntimeError(f'Expected {N_UNIVERSAL_CANDIDATES * len(ELEMENTS)} element checkpoints; found {len(universal_val)}')
universal_val.to_csv(OUTPUT_DIR / 'universal_candidate_validation.csv', index=False)
selected_universal = (
    universal_val.sort_values(['element', 'val_eta'], ascending=[True, False])
    .drop_duplicates('element', keep='first').reset_index(drop=True)
)
selected_universal.to_csv(OUTPUT_DIR / 'selected_universal_by_element.csv', index=False)
selected_paths = {row.element: Path(row.checkpoint) for row in selected_universal.itertuples()}

# Candidate landscape summarizes validation only, averaged over elements and seeds.
universal_grid_summary = (
    universal_val.groupby(['loss_balance_power', 'dropout', 'learning_rate'], as_index=False)
    .agg(mean_relative_median_mse=('val_relative_median_mse', 'mean'), mean_eta=('val_eta', 'mean'))
)
universal_grid_summary.to_csv(OUTPUT_DIR / 'universal_grid_validation_summary.csv', index=False)
display(selected_universal[['element', 'experiment_id', 'epoch', 'loss_balance_power', 'dropout', 'learning_rate', 'val_median_mse', 'val_eta']])
print(f'Completed {N_UNIVERSAL_CANDIDATES} UniversalXAS configurations.')


## Fine-tuning search: adaptation mode and learning rate

For each element, fine-tune the three strongest UniversalXAS sources by validation eta. Try four adaptations per source: update the whole head at `1e-4` or `3e-4`, or update only the final spectrum-output layer at `3e-4` or `1e-3`. Output-layer-only tuning freezes the hidden layers and batch-normalization state. Select the winning tuned model per element using validation eta; test data remain untouched until the last comparison.


In [ ]:
def train_tuned_head(element, source_checkpoint, tune_seed, strategy, learning_rate):
    source = torch.load(source_checkpoint, map_location='cpu', weights_only=False)
    head = XASSpectralHead(dropout=source.get('dropout', 0.10)).to(DEVICE)
    head.load_state_dict(source['state_dict'], strict=True)
    set_seed(tune_seed)
    if strategy == 'full_head':
        trainable_parameters = list(head.parameters())
        for parameter in trainable_parameters:
            parameter.requires_grad_(True)
    elif strategy == 'output_layer_only':
        for parameter in head.parameters():
            parameter.requires_grad_(False)
        output_layer = head[-2]
        for parameter in output_layer.parameters():
            parameter.requires_grad_(True)
        trainable_parameters = list(output_layer.parameters())
    else:
        raise ValueError(f'Unknown fine-tuning strategy: {strategy}')

    out_dir = (OUTPUT_DIR / 'tuned_candidates' / f'{element}_FEFF'
               / source['experiment_id'] / f'{strategy}_lr{short_float(learning_rate)}')
    out_dir.mkdir(parents=True, exist_ok=False)
    checkpoint = out_dir / 'best.pt'
    optimizer = torch.optim.Adam(trainable_parameters, lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    item_train, item_val = data[(element, 'train')], data[(element, 'val')]
    train_x = torch.as_tensor(item_train['X'], dtype=torch.float32, device=DEVICE)
    train_y = torch.as_tensor(item_train['y'], dtype=torch.float32, device=DEVICE)
    val_x = torch.as_tensor(item_val['X'], dtype=torch.float32, device=DEVICE)
    val_y = torch.as_tensor(item_val['y'], dtype=torch.float32, device=DEVICE)
    scale = LOSS_SCALES[element]
    best_val_relative, stale = float('inf'), 0
    history = []
    progress = tqdm(range(FINE_TUNE_MAX_EPOCHS), desc=f'Tune {element} {strategy} from {source["experiment_id"]}', unit='epoch')
    for epoch in progress:
        if strategy == 'full_head':
            head.train()
        else:
            head.eval()  # freeze BatchNorm running statistics and disable dropout
        order = torch.randperm(len(train_x), device=DEVICE)
        for batch_indices in order.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            loss = (head(train_x[batch_indices]) - train_y[batch_indices]).square().mean() / scale
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            val_errors = (head(val_x) - val_y).square().mean(dim=1).cpu().numpy()
        val_median = float(np.median(val_errors))
        relative = val_median / scale
        history.append({'epoch': epoch + 1, 'val_median_mse': val_median, 'val_relative_median_mse': relative})
        if relative < best_val_relative:
            best_val_relative, stale = relative, 0
            val_baseline = float(np.median(np.mean((item_val['y'] - TRAIN_MEANS[element]) ** 2, axis=1)))
            torch.save({
                'state_dict': cpu_state_dict(head), 'seed': tune_seed,
                'source_seed': source['seed'], 'source_epoch': source['epoch'],
                'source_experiment_id': source['experiment_id'],
                'source_loss_balance_power': source['loss_balance_power'],
                'dropout': source.get('dropout', 0.10), 'strategy': strategy,
                'learning_rate': learning_rate, 'epoch': epoch + 1,
                'val_median_mse': val_median, 'val_eta': val_baseline / max(val_median, 1e-12),
            }, checkpoint)
        else:
            stale += 1
        scheduler.step(relative)
        progress.set_postfix(val_rel=f'{relative:.4f}', best=f'{best_val_relative:.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.1e}')
        if stale >= FINE_TUNE_PATIENCE:
            break
    progress.close()
    pd.DataFrame(history).to_csv(out_dir / 'validation_history.csv', index=False)
    return checkpoint


# Select source models on validation, then compare four fine-tuning choices per source.
top_universal = (
    universal_val.sort_values(['element', 'val_eta'], ascending=[True, False])
    .groupby('element', sort=False).head(TOP_UNIVERSAL_CANDIDATES_PER_ELEMENT)
    .reset_index(drop=True)
)
top_universal.to_csv(OUTPUT_DIR / 'top_universal_candidates_for_fine_tuning.csv', index=False)
if len(top_universal) != len(ELEMENTS) * TOP_UNIVERSAL_CANDIDATES_PER_ELEMENT:
    raise ValueError('Not enough UniversalXAS candidates for the requested per-element fine-tuning sweep.')

tuned_candidate_records = []
tune_run_index = 0
for row in top_universal.itertuples():
    element = row.element
    for tune_config in FINE_TUNE_CONFIGS:
        tune_seed = 100_000 + tune_run_index
        tune_run_index += 1
        checkpoint = train_tuned_head(
            element, Path(row.checkpoint), tune_seed,
            strategy=tune_config['strategy'], learning_rate=tune_config['learning_rate'],
        )
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        tuned_candidate_records.append({
            'element': element, 'source_seed': saved['source_seed'],
            'source_experiment_id': saved['source_experiment_id'],
            'source_epoch': saved['source_epoch'],
            'source_loss_balance_power': saved['source_loss_balance_power'],
            'strategy': saved['strategy'], 'learning_rate': saved['learning_rate'],
            'tune_seed': saved['seed'], 'fine_tune_epoch': saved['epoch'],
            'val_median_mse': saved['val_median_mse'], 'val_eta': saved['val_eta'],
            'checkpoint': str(checkpoint),
        })

tuned_val = pd.DataFrame(tuned_candidate_records)
if len(tuned_val) != N_FINE_TUNE_CANDIDATES:
    raise RuntimeError(f'Expected {N_FINE_TUNE_CANDIDATES} fine-tuned candidates; found {len(tuned_val)}')
tuned_val.to_csv(OUTPUT_DIR / 'tuned_candidate_validation.csv', index=False)
selected_tuned = (
    tuned_val.sort_values(['element', 'val_eta'], ascending=[True, False])
    .drop_duplicates('element', keep='first').reset_index(drop=True)
)
selected_tuned.to_csv(OUTPUT_DIR / 'selected_tuned_by_element.csv', index=False)
tuned_paths = {row.element: Path(row.checkpoint) for row in selected_tuned.itertuples()}
display(selected_tuned[['element', 'source_experiment_id', 'strategy', 'learning_rate', 'fine_tune_epoch', 'val_median_mse', 'val_eta']])
print(f'Completed {N_FINE_TUNE_CANDIDATES} fine-tuning configurations.')


## Final validation/test tables and experiment diagnostics

The final comparison is two compact eta tables (eight elements plus a mean row) and paired validation/test bar charts. A separate validation-only heatmap summarizes the 81 UniversalXAS settings; this helps explain which loss balance, dropout, and learning rate performed well without looking at test scores. The full candidate results and detailed metrics are saved to CSV for follow-up analysis.


In [ ]:
def load_head(checkpoint):
    saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
    model = XASSpectralHead(dropout=saved.get('dropout', 0.10)).to(DEVICE)
    model.load_state_dict(saved['state_dict'], strict=True)
    return model.eval()


def score_on_split(element, variant, split, checkpoint):
    head = load_head(checkpoint)
    errors = evaluate_element(head, element, split)
    y = data[(element, split)]['y']
    baseline_errors = np.mean((y - TRAIN_MEANS[element]) ** 2, axis=1)
    baseline_median = float(np.median(baseline_errors))
    median_mse = float(np.median(errors))
    return {
        'element': element, 'variant': variant, 'evaluation_split': split,
        'n_spectra': len(errors), 'aggregate_mse': float(errors.mean()),
        'median_spectrum_mse': median_mse, 'baseline_median_mse': baseline_median,
        'eta': baseline_median / max(median_mse, 1e-12),
    }


comparison_rows = []
for element in ELEMENTS:
    candidates = [
        ('Existing UniversalXAS', OLD_UNIVERSAL),
        ('Existing Tuned-UniversalXAS', OLD_TUNED[element]),
        ('Sweep-selected UniversalXAS', selected_paths[element]),
        ('Sweep-selected Tuned-UniversalXAS', tuned_paths[element]),
    ]
    for variant, checkpoint in candidates:
        for split in ('val', 'test'):
            comparison_rows.append(score_on_split(element, variant, split, checkpoint))
comparison = pd.DataFrame(comparison_rows).sort_values(['element', 'variant', 'evaluation_split'])
comparison.to_csv(OUTPUT_DIR / 'comparison_validation_and_test.csv', index=False)

VARIANT_ORDER = [
    'Existing UniversalXAS',
    'Existing Tuned-UniversalXAS',
    'Sweep-selected UniversalXAS',
    'Sweep-selected Tuned-UniversalXAS',
]
DISPLAY_NAMES = {
    'Existing UniversalXAS': 'Existing UniversalXAS',
    'Existing Tuned-UniversalXAS': 'Existing Tuned-UniversalXAS',
    'Sweep-selected UniversalXAS': 'Sweep UniversalXAS',
    'Sweep-selected Tuned-UniversalXAS': 'Sweep Tuned-UniversalXAS',
}


def eta_summary(split):
    split_rows = comparison[comparison.evaluation_split == split]
    eta_wide = split_rows.pivot(index='element', columns='variant', values='eta').reindex(ELEMENTS)
    eta_wide = eta_wide[VARIANT_ORDER].rename(columns=DISPLAY_NAMES)
    counts = (split_rows[split_rows.variant == VARIANT_ORDER[0]]
              .set_index('element')['n_spectra'].reindex(ELEMENTS))
    table = eta_wide.copy()
    table.insert(0, 'n_spectra', counts.astype(int))
    mean_row = {column: float(table[column].mean()) for column in table.columns if column != 'n_spectra'}
    mean_row['n_spectra'] = int(table['n_spectra'].sum())
    table.loc['Mean across elements'] = mean_row
    table.index.name = 'element'
    table = table.reset_index()
    table.to_csv(OUTPUT_DIR / f'eta_{split}_summary.csv', index=False)
    return table


validation_eta_table = eta_summary('val')
test_eta_table = eta_summary('test')
print('Validation eta (higher is better)')
display(validation_eta_table.round(3))
print('Test eta (higher is better; final held-out comparison)')
display(test_eta_table.round(3))

import matplotlib.pyplot as plt

# Validation-only experiment landscape, averaged over all elements and seeds.
heatmap_data = universal_grid_summary.pivot(
    index='loss_balance_power', columns='dropout', values='mean_relative_median_mse'
).sort_index().sort_index(axis=1)
fig, axes = plt.subplots(1, len(UNIVERSAL_LEARNING_RATES), figsize=(13, 4), sharey=True)
vmin, vmax = heatmap_data.to_numpy().min(), heatmap_data.to_numpy().max()
for ax, learning_rate in zip(axes, UNIVERSAL_LEARNING_RATES):
    subset = (universal_grid_summary[universal_grid_summary.learning_rate == learning_rate]
              .pivot(index='loss_balance_power', columns='dropout', values='mean_relative_median_mse')
              .reindex(index=heatmap_data.index, columns=heatmap_data.columns))
    image = ax.imshow(subset.to_numpy(), cmap='viridis_r', aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_title(f'LR={learning_rate:g}')
    ax.set_xticks(range(len(subset.columns)), [f'{v:g}' for v in subset.columns])
    ax.set_yticks(range(len(subset.index)), [f'{v:g}' for v in subset.index])
    ax.set_xlabel('Dropout')
    for row_i in range(subset.shape[0]):
        for col_i in range(subset.shape[1]):
            ax.text(col_i, row_i, f'{subset.iloc[row_i, col_i]:.3f}', ha='center', va='center', color='white')
axes[0].set_ylabel('Loss balance power (alpha)')
fig.colorbar(image, ax=axes, label='Mean validation relative median MSE (lower is better)', shrink=0.85)
fig.suptitle('UniversalXAS validation search landscape', y=1.03)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'universal_validation_grid_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

# Final eta comparison: same four model variants, shown separately for validation and test.
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, split, title in [
    (axes[0], 'val', 'Validation eta'),
    (axes[1], 'test', 'Test eta'),
]:
    split_rows = comparison[comparison.evaluation_split == split]
    eta_plot = (split_rows.pivot(index='element', columns='variant', values='eta')
                .reindex(index=ELEMENTS, columns=VARIANT_ORDER)
                .rename(columns=DISPLAY_NAMES))
    eta_plot.plot(kind='bar', ax=ax, rot=0, width=0.82)
    ax.set_title(title)
    ax.set_xlabel('Element')
    ax.set_ylabel('Eta (higher is better)')
    ax.grid(axis='y', alpha=0.25)
    ax.legend_.remove()
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08))
fig.suptitle('Existing models vs validation-selected experiment sweep', y=1.16)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'validation_test_eta_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'Validation tables, charts, candidates, checkpoints, and detailed metrics saved in {OUTPUT_DIR}')
